## Instalação das bibliotecas

In [ ]:
pip install python-dotenv

In [ ]:
pip install pyspark boto3

## Configuração das credenciais AWS e da sessão Spark

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_unixtime, avg, count
from dotenv import load_dotenv

In [ ]:
load_dotenv('.env_kafka_connect')

aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = "us-east-1"
s3_bucket = "bucket-geral-do-df-01"

In [ ]:
current_dir = os.getcwd()
hadoop_aws_jar = os.path.join(current_dir, "hadoop-aws-3.3.4.jar")
aws_sdk_jar = os.path.join(current_dir, "aws-java-sdk-bundle-1.12.262.jar")
jars_path = f"{hadoop_aws_jar},{aws_sdk_jar}"

spark = SparkSession.builder \
    .appName("ETL Pipeline - S3 Integration") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars", jars_path) \
    .getOrCreate()

spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", aws_access_key)
spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", aws_secret_key)
spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", f"s3.{aws_region}.amazonaws.com")
spark._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "true")
spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")

## Camada Bronze — leitura dos dados brutos do S3

In [ ]:
bronze_path = f"s3a://{s3_bucket}/raw-data/ipca/kafka/"

try:
    df_bronze = spark.read.json(bronze_path)
    print("Leitura bem-sucedida!")
    df_bronze.show()
except Exception as e:
    print(f"Erro ao acessar o S3: {e}")

## Camada Silver — limpeza e padronização

In [ ]:
df_silver = df_bronze.dropDuplicates()

df_silver = df_silver.withColumn("Data_Vencimento", from_unixtime(col("Data_Vencimento") / 1000, "yyyy-MM-dd")) \
                     .withColumn("Data_Base", from_unixtime(col("Data_Base") / 1000, "yyyy-MM-dd")) \
                     .withColumn("dt_update", from_unixtime(col("dt_update") / 1000, "yyyy-MM-dd HH:mm:ss"))

df_silver = df_silver.fillna({
    "PUCompraManha": 0,
    "PUVendaManha": 0,
    "PUBaseManha": 0
})

print("Dados Transformados (Silver):")
df_silver.show(truncate=False)

silver_path = f"s3a://{s3_bucket}/processed-data/ipca/silver/"
df_silver.write.mode("overwrite").parquet(silver_path)

## Camada Gold — métricas agregadas

In [ ]:
df_gold = df_silver.groupBy("Tipo").agg(
    avg("PUCompraManha").alias("Media_PUCompraManha"),
    avg("PUVendaManha").alias("Media_PUVendaManha"),
    count("*").alias("Total_Registros")
)

print("Dados Agregados (Gold):")
df_gold.show(truncate=False)

gold_path = f"s3a://{s3_bucket}/analytics/ipca/gold/"
df_gold.write.mode("overwrite").parquet(gold_path)

## Encerramento da sessão Spark

In [ ]:
spark.stop()